# Silver Layer: RapidAPI

## Imports

In [0]:
from pyspark.sql import functions as F
from silver_config import SILVER_COLUMNS

## Load From Bronze

In [0]:
rapid_df = spark.read.table("bronze.rapidapi").select(F.explode("properties").alias("p"))

## Transformations

### Casting & Normalization

In [0]:
rapid_df = rapid_df.withColumn("listing_id", F.col("p.listing_id"))
rapid_df = rapid_df.withColumn("source_url", F.col("p.permalink"))

In [0]:
rapid_df = rapid_df.withColumn("price_amount", F.col("p.list_price").try_cast("double"))
rapid_df = rapid_df.withColumn("currency", F.lit("USD"))

In [0]:
rapid_df = rapid_df.withColumn("location_city", F.col("p.location.address.city"))
rapid_df = rapid_df.withColumn("location_zip", F.col("p.location.address.postal_code"))

In [0]:
rapid_df = rapid_df.withColumn(
    "posted_date",
    F.expr("try_cast(p.list_date as timestamp)").cast("date")
)
rapid_df = rapid_df.filter(F.col("posted_date").isNotNull())

In [0]:
rapid_df = rapid_df.withColumn("rooms_count", F.col("p.description.beds").try_cast("float"))
rapid_df = rapid_df.withColumn("area", F.col("p.description.sqft").try_cast("double"))

In [0]:
rapid_df = rapid_df.withColumn("images", F.col("p.photos.href"))

In [0]:
rapid_df = rapid_df.withColumn("description", F.lit(None).cast("string"))
rapid_df = rapid_df.withColumn("title", F.lit(None).cast("string"))

In [0]:
rapid_df = rapid_df.withColumn("property_type", F.col("p.description.type"))
rapid_df = rapid_df.withColumn(
    "transaction_type",
    F.when(F.col("p.status") == "for_rent", F.lit("rent")).otherwise(F.lit("sale"))
)

In [0]:
rapid_df = rapid_df.withColumn("source", F.lit("rapidapi"))
rapid_df = rapid_df.withColumn("scraped_at", F.current_timestamp())

## Write to Silver

In [0]:
rapid_df = rapid_df.select(
    *SILVER_COLUMNS
)

In [0]:
rapid_df.write.mode("overwrite").saveAsTable("silver.rapidapi")